# Lesson 07 — Exercise Solutions

Worked solutions with a short rationale after each. Run setup.sql first.

In [ ]:
USE AdventureWorks2022;
GO

## Exercise 1

Rank all products by ListPrice descending within each subcategory. Use DENSE_RANK so there are no gaps in the ranking.
Expected columns: SubcategoryName, ProductName, ListPrice, PriceRank
Tables: Production.Product, Production.ProductSubcategory

In [ ]:
SELECT
    ps.Name  AS SubcategoryName,
    p.Name   AS ProductName,
    p.ListPrice,
    DENSE_RANK() OVER (
        PARTITION BY p.ProductSubcategoryID
        ORDER BY p.ListPrice DESC
    ) AS PriceRank
FROM Production.Product             AS p
JOIN Production.ProductSubcategory  AS ps ON ps.ProductSubcategoryID = p.ProductSubcategoryID
WHERE p.ListPrice > 0
ORDER BY SubcategoryName, PriceRank;

**Why:** PARTITION BY subcategory; ORDER BY ListPrice DESC; join for name.

## Exercise 2

For each sales order, show TotalDue and the running total of TotalDue ordered by OrderDate for ALL orders (no partition — one window for everything).
Expected columns: SalesOrderID, OrderDate, TotalDue, RunningTotal

In [ ]:
SELECT
    SalesOrderID,
    OrderDate,
    TotalDue,
    SUM(TotalDue) OVER (
        ORDER BY OrderDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS RunningTotal
FROM Sales.SalesOrderHeader
ORDER BY OrderDate;

**Why:** No PARTITION; ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW for cumsum.

## Exercise 3

Using LAG, show each order's TotalDue and the TotalDue of the PREVIOUS order for the SAME customer (ordered by OrderDate). Show NULL when there is no previous order.
Expected columns: CustomerID, SalesOrderID, OrderDate, TotalDue, PrevOrderAmount

In [ ]:
SELECT
    CustomerID,
    SalesOrderID,
    OrderDate,
    TotalDue,
    LAG(TotalDue) OVER (
        PARTITION BY CustomerID
        ORDER BY OrderDate
    ) AS PrevOrderAmount
FROM Sales.SalesOrderHeader
ORDER BY CustomerID, OrderDate;

**Why:** PARTITION BY CustomerID; LAG with no default returns NULL for first order.

## Exercise 4

Return only the MOST RECENT order for each customer (use ROW_NUMBER to pick the latest order).
Expected columns: CustomerID, SalesOrderID, OrderDate, TotalDue

In [ ]:
SELECT CustomerID, SalesOrderID, OrderDate, TotalDue
FROM (
    SELECT
        CustomerID,
        SalesOrderID,
        OrderDate,
        TotalDue,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY OrderDate DESC
        ) AS rn
    FROM Sales.SalesOrderHeader
) AS ranked
WHERE rn = 1
ORDER BY CustomerID;

**Why:** ROW_NUMBER descending by date; filter rn = 1 in outer query.

## Exercise 5

Using lesson07.MonthlySales, calculate each month's Revenue as a percentage of that region's TOTAL revenue across all months.
Expected columns: Region, SaleYear, SaleMonth, Revenue, PctOfRegionTotal

In [ ]:
SELECT
    Region,
    SaleYear,
    SaleMonth,
    Revenue,
    CAST(Revenue * 100.0
         / SUM(Revenue) OVER (PARTITION BY Region)
         AS DECIMAL(5,2)) AS PctOfRegionTotal
FROM lesson07.MonthlySales
ORDER BY Region, SaleYear, SaleMonth;

**Why:** SUM() OVER (PARTITION BY Region) with no ORDER BY = full partition sum.

## Challenges

Harder problems. They may combine this lesson's topic with
earlier lessons, and usually need 2+ concepts in one query.

### Challenge 1

Using Sales.SalesOrderHeader, find each salesperson's longest streak of consecutive months with month-over-month increasing total sales. A streak breaks on a down-or-flat month OR a gap month (a calendar month with no orders). Report each salesperson's best streak length and the months it spanned.
Expected columns: SalesPersonID, StreakLengthMonths, StreakStart, StreakEnd
Hint: Aggregate to salesperson-month first; LAG for the comparison; the difference of two ROW_NUMBERs (or a running flag-sum) groups consecutive rising months into islands — gaps-and-islands. Remember SalesPersonID is NULL for online orders.

In [ ]:
WITH Monthly AS (
    SELECT SalesPersonID,
           DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1) AS MonthStart,
           SUM(TotalDue) AS MonthTotal
    FROM Sales.SalesOrderHeader
    WHERE SalesPersonID IS NOT NULL          -- online orders have no salesperson
    GROUP BY SalesPersonID, DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1)
),
Flagged AS (
    SELECT SalesPersonID, MonthStart, MonthTotal,
        CASE WHEN LAG(MonthStart) OVER (PARTITION BY SalesPersonID ORDER BY MonthStart)
                     = DATEADD(MONTH, -1, MonthStart)                    -- no gap month
              AND MonthTotal
                     > LAG(MonthTotal) OVER (PARTITION BY SalesPersonID ORDER BY MonthStart)
             THEN 0 ELSE 1 END AS IsStreakStart
    FROM Monthly
),
Grouped AS (
    SELECT SalesPersonID, MonthStart,
        SUM(IsStreakStart) OVER (PARTITION BY SalesPersonID ORDER BY MonthStart
                                 ROWS UNBOUNDED PRECEDING) AS StreakId
    FROM Flagged
),
Streaks AS (
    SELECT SalesPersonID, StreakId,
           COUNT(*)        AS StreakLengthMonths,
           MIN(MonthStart) AS StreakStart,
           MAX(MonthStart) AS StreakEnd
    FROM Grouped
    GROUP BY SalesPersonID, StreakId
)
SELECT SalesPersonID, StreakLengthMonths, StreakStart, StreakEnd
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY SalesPersonID
                 ORDER BY StreakLengthMonths DESC, StreakStart) AS rn
    FROM Streaks
) AS best
WHERE rn = 1                                  -- one longest streak per salesperson
ORDER BY StreakLengthMonths DESC, SalesPersonID;

**Why:** Decompose: (a) collapse orders to salesperson-month totals, (b) flag
where a rising run starts, (c) group each run, (d) keep the longest. A month
continues a streak only if LAG shows the immediately prior calendar month AND a
lower total — the tempting-but-wrong approach LAGs on value alone, silently
treating gap months as consecutive and over-counting. A running SUM of the start
flag is a group id (gaps-and-islands); COUNT is the length, MIN/MAX the span.

### Challenge 2

For each customer with 5 or more orders: their top-3 orders by TotalDue, and each of those orders' percentage of that customer's lifetime spend.
Expected columns: CustomerID, SalesOrderID, TotalDue, PctOfLifetimeSpend, OrderRank

In [ ]:
WITH Ranked AS (
    SELECT CustomerID, SalesOrderID, TotalDue,
        COUNT(*)      OVER (PARTITION BY CustomerID) AS OrderCount,
        SUM(TotalDue) OVER (PARTITION BY CustomerID) AS LifetimeSpend,
        ROW_NUMBER()  OVER (PARTITION BY CustomerID ORDER BY TotalDue DESC) AS OrderRank
    FROM Sales.SalesOrderHeader
)
SELECT CustomerID, SalesOrderID, TotalDue,
       CAST(TotalDue * 100.0 / LifetimeSpend AS DECIMAL(5,2)) AS PctOfLifetimeSpend,
       OrderRank
FROM Ranked
WHERE OrderCount >= 5 AND OrderRank <= 3
ORDER BY CustomerID, OrderRank;

**Why:** One pass computes three windows over the same PARTITION BY CustomerID:
COUNT(*) for the 5+ gate, SUM(TotalDue) for the lifetime denominator, ROW_NUMBER
(TotalDue DESC) for the top-3. All three are windows, so detail rows survive with
no GROUP BY; filtering happens in the outer query because window functions are
illegal in WHERE. Tempting-but-wrong: GROUP BY for the count/sum plus a self-join
back for the top 3 — more work, and it collapses the rows you still need.

### Challenge 3

Using lesson07.MonthlySales, compute each region's 3-month moving average of revenue, plus a flag column marking months where revenue fell more than 20% below that moving average.
Expected columns: Region, SaleYear, SaleMonth, Revenue, MovAvg3, IsSlumpMonth

In [ ]:
WITH M AS (
    SELECT Region, SaleYear, SaleMonth, Revenue,
        AVG(Revenue) OVER (PARTITION BY Region ORDER BY SaleYear, SaleMonth
                           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS MovAvg3
    FROM lesson07.MonthlySales
)
SELECT Region, SaleYear, SaleMonth, Revenue,
       CAST(MovAvg3 AS DECIMAL(14,2)) AS MovAvg3,
       CASE WHEN Revenue < 0.8 * MovAvg3 THEN 1 ELSE 0 END AS IsSlumpMonth
FROM M
ORDER BY Region, SaleYear, SaleMonth;

**Why:** Compute the moving average in a CTE (frame ROWS BETWEEN 2 PRECEDING AND
CURRENT ROW, per region), then compare Revenue to it in the outer query — a SELECT
column alias can't be referenced by another expression in the same SELECT, so the
flag needs the CTE layer. Tempting-but-wrong: repeating the whole AVG() OVER inside
the CASE — legal, but duplicated logic that drifts on edit. "20% below" = Revenue
< 0.8 * MovAvg3; West's month-4 dip (12000 vs ~19333) is the row that trips it.

### Challenge 4

Split all customers into spending quartiles with NTILE(4) over lifetime TotalDue, then report one row per quartile: customer count and the min / avg / max lifetime spend in that quartile.
Expected columns: SpendQuartile, CustomerCount, MinSpend, AvgSpend, MaxSpend

In [ ]:
WITH Lifetime AS (
    SELECT CustomerID, SUM(TotalDue) AS LifetimeSpend
    FROM Sales.SalesOrderHeader
    GROUP BY CustomerID
),
Quartiled AS (
    SELECT CustomerID, LifetimeSpend,
        NTILE(4) OVER (ORDER BY LifetimeSpend) AS SpendQuartile
    FROM Lifetime
)
SELECT SpendQuartile,
       COUNT(*)                                  AS CustomerCount,
       CAST(MIN(LifetimeSpend) AS DECIMAL(14,2)) AS MinSpend,
       CAST(AVG(LifetimeSpend) AS DECIMAL(14,2)) AS AvgSpend,
       CAST(MAX(LifetimeSpend) AS DECIMAL(14,2)) AS MaxSpend
FROM Quartiled
GROUP BY SpendQuartile
ORDER BY SpendQuartile;

**Why:** Two window layers separated by a grain change. First collapse orders to
one lifetime total per customer (GROUP BY), THEN apply NTILE(4) over that ordered
list to bucket customers. NTILE must run on the per-customer totals, not raw order
rows — bucketing raw orders would quartile transactions, not customers. Finally a
plain GROUP BY on the quartile label rolls each bucket up to count/min/avg/max.